# Lesson 9 | How can one compute unit serve many neurons?

So far one RTL neuron has held its own `membrane_v`. A large system does not have to duplicate a complete compute datapath for every virtual neuron. Today asks:
> **How can one physical compute unit take turns updating many virtual neurons?**

Primary concept: **time multiplexing**.


## 1. Concept ledger

**Known:** one neuron update, combinational path, register state, and clocked update.

**New:** time multiplexing; supporting terms are memory, address, and RAM.

**Preview:** spike queues, sparse synapses, and routing come in the next three lessons.


## 2. Why not simply duplicate one neuron engine per neuron?

More compute units can provide more parallelism, but they also consume more logic and storage resources. Another architecture stores many neuron states in memory and lets a smaller number of compute engines read, update, and write them back in turn.

The key idea is that the number of **physical compute engines** need not equal the number of **virtual neurons**.


## 3. Three supporting terms

**memory:** storage for many pieces of state.

**address:** a number selecting which memory location is read or written.

**Random-Access Memory (RAM):** memory whose locations are selected by address. For now, treat it as many numbered state slots; FPGA BRAM/URAM details come later.


## 4. The core of time multiplexing

Time multiplexing means **the same physical compute unit serves different virtual objects at different times.**

```mermaid
flowchart LR
 A["address / neuron_id"] --> MEM["state memory"]
 MEM --> ENG["one neuron engine"]
 ENG --> MEM
 SCHED["scheduler: 0,1,2,3,..."] --> A
```

The important idea is not the Python `for` loop itself, but the architecture: select state by address → update with one engine → write back to the same address.


## 5. Run: four virtual neurons, one update process

A Python list stands in for state memory. Each address changes only when it is served. Predict the final memory first.


In [ ]:
states = [0, 10, -3, 7]
inputs = [2, -1, 4, 0]

print('address | before | input | after')
for address, input_value in enumerate(inputs):
    before = states[address]
    after = before + input_value
    states[address] = after
    print(f'{address:7d} | {before:6d} | {input_value:5d} | {after:5d}')

print('final memory:', states)


## 6. Observe

Look for two properties:

1. Updating address 0 must not alter other addresses.
2. After one pass, all four states have been served by the same update rule.

This is the minimal intuition behind a later `MOD-004 neuron_state_store` plus scheduler.


## 7. Try It: trading resources for time

Assume 8 neurons and one update slot per neuron.

- With 1 engine, how many update slots are needed for one round?
- With 2 engines, how could the addresses be divided ideally?

Answer first, then modify the Python loop so one virtual engine handles even addresses and another handles odd addresses. Ignore real pipeline latency for now.


## 8. Homework

Open:

[Lesson 9 Exercise: One engine serving many states](../../exercises/en/09_time_multiplex_many_neurons.ipynb)

You will implement addressed state updates and round-robin service, then call the external grader directly on the functions in the current Notebook kernel.

## 9. AI Task

Ask AI to compare “four neurons with four engines” against “four virtual neurons time-multiplexed through one engine.” Require a qualitative resource/time comparison only; it must not invent LUT, MHz, or power numbers.


## 10. Human Check

Without AI, explain why an address is not neuron state, what memory stores, how one engine serves many virtual neurons, and why time multiplexing usually saves compute resources while taking more time to complete a full round.


## 11. Engineering Handoff

This lesson builds intuition for `MOD-004 neuron_state_store` and RMD-006/007. It does not freeze RAM timing, banking, the scheduler, or the formal neuron interface.


## 12. Project Trace

- Lesson: `LSN-009`
- Mapping: `RMD-006 / RMD-007` teaching precursor
- Module context: `MOD-004`
- Formal implementation status: not complete


## 13. Exit Ticket

You can draw `address → state memory → one engine → write back` and explain why “many virtual neurons” does not imply “many physical engines.”
